# Original-Model Replication (Leakage-Safe, Original Data)

Replicates the final hand-picked model from `Supervised-Learning-BTC-Price-Direction/Pricing
Models/ML-MODELS.ipynb`, but now sourced from that same repo's own CSVs (`bitcoin-price.csv`,
`market-data.csv`, `on-chain-data.csv`, `economic-more.csv`) instead of freshly-pulled
yfinance/FRED/CoinMetrics substitutes. Rebuilt with the leakage-safety standards developed
across `BTC-mk4.ipynb` and `btc-direcitonal.ipynb` this session, and extended with the
original notebook's own broader model comparison (Logistic Regression, Random Forest, LSTM,
CNN-LSTM).

## What's replicated

All 14 of the original's final features, using the original's own data and transforms:

| Original feature | Source column | Transform |
|---|---|---|
| S&P500, Nasdaq, Dow, Gold, Silver, Copper | `market-data.csv` | `pct_change()` |
| Fed Funds Rate (raw + 1d delta) | `economic-more.csv` (`FDTR Index`) | raw, `diff(1)` |
| 10Y Treasury (pct change + 5d delta) | `economic-more.csv` (`USGG10YR Index`) | `pct_change()`, `diff(5)` |
| Hashrate | `on-chain-data.csv` | 7d rolling mean |
| Miner Reserve | `on-chain-data.csv` | `diff(7)` |
| Mean Coin Age | `on-chain-data.csv` | `pct_change(7)` |
| Open Interest | `on-chain-data.csv` | `pct_change()` |

Every source is inner-joined on date (matching the original notebook's own merge strategy),
so every feature comes from the exact same overlapping calendar -- no feature is silently
extrapolated or forward-filled across a range another feature doesn't cover.

**Still fixed from the original:** the original used a single last-fold `TimeSeriesSplit`
holdout with no significance test, and did no leakage audit on its FRED-equivalent series.
Here every evaluation is a proper chronological walk-forward (8 expanding folds) with
McNemar's test and a fold-clustered bootstrap CI against an **Always-up** baseline (predict
"bullish" every day).

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
pd.set_option("display.max_columns", 50)

ORIG_DATA_DIR = Path("../data/original")
MK4_DIR = Path("../data/mk4")
print("Imports OK. Torch:", torch.__version__)


Imports OK. Torch: 2.12.1


## 1. Load and Merge All Data Sources

Loads the original notebook's own CSVs directly (`bitcoin-price.csv`, `market-data.csv`,
`on-chain-data.csv`, `economic-more.csv`) rather than freshly-pulled substitute sources.
Every source is inner-joined on `date`, so the merged table -- and every feature built from
it -- is confined to the single overlapping timeframe all four sources actually share.

In [2]:
btc_raw = pd.read_csv(ORIG_DATA_DIR / "bitcoin-price.csv")
market_raw = pd.read_csv(ORIG_DATA_DIR / "market-data.csv")
onchain_raw = pd.read_csv(ORIG_DATA_DIR / "on-chain-data.csv")
econ_more_raw = pd.read_csv(ORIG_DATA_DIR / "economic-more.csv")

btc_raw["date"] = pd.to_datetime(btc_raw["Datetime"]).dt.tz_localize(None).dt.normalize()
market_raw["date"] = pd.to_datetime(market_raw["Dates"])
onchain_raw["date"] = pd.to_datetime(onchain_raw["Datetime"]).dt.tz_localize(None).dt.normalize()
econ_more_raw["date"] = pd.to_datetime(econ_more_raw["Dates"])

btc = (btc_raw[["date", "Close"]]
       .rename(columns={"Close": "btc_close"})
       .dropna(subset=["btc_close"]))

market = market_raw.rename(columns={
    "SPX Index": "SP500", "CCMP Index": "Nasdaq", "INDU Index": "Dow",
    "XAU Curncy": "Gold", "XAG Curncy": "Silver", "LMCADY Comdty": "Copper",
})[["date", "SP500", "Nasdaq", "Dow", "Gold", "Silver", "Copper"]]

onchain = onchain_raw.rename(columns={
    "Miner Reserve": "miner_reserve", "Hashrate": "hashrate",
    "Mean Coin Age": "mean_coin_age", "Open Interest": "open_interest",
})[["date", "miner_reserve", "hashrate", "mean_coin_age", "open_interest"]]

econ = econ_more_raw.rename(columns={
    "FDTR Index": "Fed Funds Rate", "USGG10YR Index": "10Y Treasury Yield",
})[["date", "Fed Funds Rate", "10Y Treasury Yield"]]

# Inner-join every source on date (matching the original notebook's own merge strategy):
# every feature is confined to the exact same overlapping timeframe, no ffill across
# mismatched calendars (market-data.csv is weekdays-only, on-chain-data.csv is every day).
raw = (btc.merge(market, on="date", how="inner")
          .merge(onchain, on="date", how="inner")
          .merge(econ, on="date", how="inner"))
raw = raw.dropna().sort_values("date").reset_index(drop=True)

raw_start = raw["date"].min().date()
raw_end = raw["date"].max().date()
print(f"Merged raw table: {raw.shape}, {raw_start} to {raw_end}")
raw.tail()


Merged raw table: (1616, 14), 2019-04-01 to 2025-06-09


,date,btc_close,SP500,Nasdaq,Dow,Gold,Silver,Copper,miner_reserve,hashrate,mean_coin_age,open_interest,Fed Funds Rate,10Y Treasury Yield
1611,2025-06-03,105391.152971,5970.37,19398.96,42519.64,3353.43,34.5176,9686.31,1.808612e+06,9.089746e+11,1614.212233,3.547176e+10,4.5,4.4537
1612,2025-06-04,104724.591994,5970.81,19460.49,42427.74,3372.72,34.5045,9669.98,1.807733e+06,7.953528e+11,1613.870778,3.474156e+10,4.5,4.3552
1613,2025-06-05,101613.260806,5939.30,19298.45,42319.74,3352.65,35.6510,9832.65,1.807730e+06,8.458514e+11,1614.501350,3.385101e+10,4.5,4.3906
1614,2025-06-06,104406.924809,6000.36,19529.95,42762.87,3310.42,35.9777,9762.84,1.807638e+06,8.458514e+11,1615.110946,3.543465e+10,4.5,4.5056
1615,2025-06-09,110303.871227,6005.88,19591.24,42761.76,3326.19,36.7584,9888.78,1.808101e+06,8.837253e+11,1617.120227,3.784251e+10,4.5,4.4738


## 2. Feature Transforms

Replicates the original notebook's exact transform choices for all 14 of its final
hand-picked features (`Tr X = X.pct_change()`, etc. -- see `ML-MODELS.ipynb` cell computing
`model_df`), now applied to the original's own data instead of substitutes.

In [3]:
out = raw.sort_values("date").reset_index(drop=True).copy()

# Market (replicated exactly: simple pct change, matching "Tr S&P500" etc.)
out["Tr SP500"] = out["SP500"].pct_change()
out["Tr Nasdaq"] = out["Nasdaq"].pct_change()
out["Tr Dow"] = out["Dow"].pct_change()
out["Tr Gold"] = out["Gold"].pct_change()
out["Tr Silver"] = out["Silver"].pct_change()
out["Tr Copper"] = out["Copper"].pct_change()

# Economic (replicated exactly)
out["Tr Fed Funds Raw"] = out["Fed Funds Rate"]
out["Tr Fed Funds Delta"] = out["Fed Funds Rate"] - out["Fed Funds Rate"].shift(1)
out["Tr Treasury Pct"] = out["10Y Treasury Yield"].pct_change()
out["Tr Treasury Delta"] = out["10Y Treasury Yield"] - out["10Y Treasury Yield"].shift(5)

# On-chain (replicated exactly, matching model_df's "Tr Hashrate 1", "Tr Miner Reserve 1",
# "Tr Mean Coin Age 2", "Tr Open Interest" in the original notebook)
out["Tr Hashrate"] = out["hashrate"].rolling(7).mean()
out["Tr Miner Reserve"] = out["miner_reserve"] - out["miner_reserve"].shift(7)
out["Tr Mean Coin Age"] = out["mean_coin_age"].pct_change(7)
out["Tr Open Interest"] = out["open_interest"].pct_change()

FEATURE_COLS = [c for c in out.columns if c.startswith("Tr ")]
print(f"Replicated features ({len(FEATURE_COLS)}):")
for f in FEATURE_COLS:
    print(" ", f)


Replicated features (14):
  Tr SP500
  Tr Nasdaq
  Tr Dow
  Tr Gold
  Tr Silver
  Tr Copper
  Tr Fed Funds Raw
  Tr Fed Funds Delta
  Tr Treasury Pct
  Tr Treasury Delta
  Tr Hashrate
  Tr Miner Reserve
  Tr Mean Coin Age
  Tr Open Interest


## 3. Target and Chronological Frame

In [4]:
out["target_bullish_tomorrow"] = (out["btc_close"].shift(-1) > out["btc_close"]).astype("float")
out = out.iloc[:-1]  # final row's target is unknowable

model_frame = out[["date", "target_bullish_tomorrow"] + FEATURE_COLS].copy()
model_frame = model_frame.replace([np.inf, -np.inf], np.nan)
model_frame = model_frame.dropna().sort_values("date").reset_index(drop=True)
model_frame["target_bullish_tomorrow"] = model_frame["target_bullish_tomorrow"].astype(int)

model_frame.to_csv(MK4_DIR / "original_replication_model_frame.csv", index=False)
model_start = model_frame["date"].min().date()
model_end = model_frame["date"].max().date()
print(f"Model frame: {model_frame.shape}, {model_start} to {model_end}")
up_rate = model_frame["target_bullish_tomorrow"].mean()
print(f"Actual up-rate: {up_rate:.4f}")
model_frame.tail()

Model frame: (1608, 16), 2019-04-10 to 2025-06-06
Actual up-rate: 0.5118


,date,target_bullish_tomorrow,Tr SP500,Tr Nasdaq,Tr Dow,Tr Gold,Tr Silver,Tr Copper,Tr Fed Funds Raw,Tr Fed Funds Delta,Tr Treasury Pct,Tr Treasury Delta,Tr Hashrate,Tr Miner Reserve,Tr Mean Coin Age,Tr Open Interest
1603,2025-06-02,0,0.004102,0.006741,0.000838,0.028073,0.053915,0.012559,4.5,0.0,0.008976,-0.0711,9.199551e+11,272.522832,0.001853,-0.004049
1604,2025-06-03,0,0.005800,0.008125,0.005062,-0.008327,-0.007016,0.001895,4.5,0.0,0.003108,0.0101,9.158958e+11,1969.407218,0.001974,0.003840
1605,2025-06-04,0,0.000074,0.003172,-0.002161,0.005752,-0.000380,-0.001686,4.5,0.0,-0.022116,-0.1221,8.973327e+11,832.845867,0.001326,-0.020585
1606,2025-06-05,1,-0.005277,-0.008327,-0.002546,-0.005951,0.033228,0.016822,4.5,0.0,0.008128,-0.0274,8.963512e+11,450.078272,0.001877,-0.025634
1607,2025-06-06,1,0.010281,0.011996,0.010471,-0.012596,0.009164,-0.007100,4.5,0.0,0.026192,0.1052,8.754987e+11,741.735264,0.002039,0.046783


## 4. Models

Logistic Regression and Random Forest (sklearn), plus LSTM and CNN-LSTM (PyTorch), matching
the original notebook's model comparison. The two sequence models take a trailing 10-day
window of the same feature vector as input (an LSTM needs a sequence dimension; a single
day's feature row has none on its own).

In [5]:
SEQ_LOOKBACK = 10


def make_sequences(X, y, dates, lookback):
    Xs, ys, ds = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys.append(y[i])
        ds.append(dates[i])
    return np.array(Xs), np.array(ys), np.array(ds)


class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden=16):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]).squeeze(-1)


class CNNLSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden=16, conv_channels=8):
        super().__init__()
        self.conv = nn.Conv1d(n_features, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(conv_channels, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        xc = self.relu(self.conv(x.transpose(1, 2))).transpose(1, 2)
        _, (h, _) = self.lstm(xc)
        return self.fc(h[-1]).squeeze(-1)


def train_torch_classifier(model_cls, Xtr, ytr, n_features, epochs=15, lr=0.01, batch_size=64, seed=42):
    torch.manual_seed(seed)
    model = model_cls(n_features=n_features)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32)
    n = len(Xtr_t)
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            logits = model(Xtr_t[idx])
            loss = loss_fn(logits, ytr_t[idx])
            loss.backward()
            opt.step()
    model.eval()
    return model


def torch_predict_proba(model, X):
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32))
        return torch.sigmoid(logits).numpy()


print("Model classes defined")

Model classes defined


## 5. Walk-Forward Evaluation (All 4 Models, 8 Chronological Folds)

`TimeSeriesSplit`, expanding training window, exactly the pattern validated across both other
notebooks this session. The two sequence models additionally need a `SEQ_LOOKBACK`-day warmup
within each fold before their first prediction (the first 10 rows of every fold's test period
have no full lookback window from *that fold's* training data alone, so they're built from the
tail of the training data as normal sequence context -- still only past information).

The Always-up baseline is computed on the same test rows for every fold.

**Purge:** the last training row before each fold's test split has a label built from
`close.shift(-1)` -- i.e. tomorrow's price relative to today's, where "tomorrow" is the first
test-period date. That one row's label reaches one day into the test period before the model
ever sees a test row. At this notebook's 1-day horizon that's a single row per fold (out of
~200-1400), which is the same purge applied at the 7-day and 30-day horizons in
`original_model_replication_multiday.ipynb` -- there it drops up to 29 rows per fold and
meaningfully changes results; here it's applied for methodological consistency, not because
one row per fold is expected to move anything.

In [6]:
X_all = model_frame[FEATURE_COLS].to_numpy()
y_all = model_frame["target_bullish_tomorrow"].to_numpy()
dates_all = model_frame["date"].to_numpy()

HORIZON_DAYS = 1  # target is close.shift(-1) > close

tscv = TimeSeriesSplit(n_splits=8)
results_rows = []
fold_num = 0

for train_idx_raw, test_idx in tscv.split(X_all):
    fold_num += 1

    # Purge: drop training rows whose label (HORIZON_DAYS ahead) reaches into the test
    # period -- otherwise those rows' labels already encode test-period price movement.
    purge_mask = (train_idx_raw + HORIZON_DAYS) >= test_idx.min()
    train_idx = train_idx_raw[~purge_mask]
    n_purged = len(train_idx_raw) - len(train_idx)

    Xtr_raw, ytr = X_all[train_idx], y_all[train_idx]
    Xte_raw, yte = X_all[test_idx], y_all[test_idx]
    dates_te = dates_all[test_idx]

    scaler = StandardScaler().fit(Xtr_raw)
    Xtr = scaler.transform(Xtr_raw)
    Xte = scaler.transform(Xte_raw)

    # --- Logistic Regression ---
    logit = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    proba_logit = logit.predict_proba(Xte)[:, 1]

    # --- Random Forest ---
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    proba_rf = rf.predict_proba(Xte)[:, 1]

    # --- LSTM / CNN-LSTM: sequence data spans the train/test boundary using only
    # already-known (training-fold) history for the earliest test-fold sequences.
    X_fold_full = np.concatenate([Xtr, Xte])
    y_fold_full = np.concatenate([ytr, yte])
    Xseq, yseq, _ = make_sequences(X_fold_full, y_fold_full, np.arange(len(X_fold_full)), SEQ_LOOKBACK)
    n_test = len(Xte)
    # sequences whose target index falls within the test region
    seq_target_idx = np.arange(SEQ_LOOKBACK, len(X_fold_full))
    test_seq_mask = seq_target_idx >= len(Xtr)
    Xseq_tr, yseq_tr = Xseq[~test_seq_mask], yseq[~test_seq_mask]
    Xseq_te, yseq_te = Xseq[test_seq_mask], yseq[test_seq_mask]

    lstm_model = train_torch_classifier(LSTMClassifier, Xseq_tr, yseq_tr, n_features=Xtr.shape[1])
    proba_lstm = torch_predict_proba(lstm_model, Xseq_te)

    cnn_lstm_model = train_torch_classifier(CNNLSTMClassifier, Xseq_tr, yseq_tr, n_features=Xtr.shape[1])
    proba_cnn_lstm = torch_predict_proba(cnn_lstm_model, Xseq_te)

    # --- Baseline ---
    always_up = np.ones(len(yte), dtype=int)

    for i in range(len(yte)):
        results_rows.append({
            "date": pd.Timestamp(dates_te[i]), "fold": fold_num, "actual": int(yte[i]),
            "proba_logit": proba_logit[i], "proba_rf": proba_rf[i],
            "proba_lstm": proba_lstm[i], "proba_cnn_lstm": proba_cnn_lstm[i],
            "always_up_pred": always_up[i],
        })
    print(f"Fold {fold_num}: train={len(ytr)} (purged {n_purged})  test={len(yte)}  test_seq={len(yseq_te)}  "
          f"({pd.Timestamp(dates_te[0]).date()} to {pd.Timestamp(dates_te[-1]).date()})")

daily = pd.DataFrame(results_rows)
for m in ["logit", "rf", "lstm", "cnn_lstm"]:
    daily[f"pred_{m}"] = (daily[f"proba_{m}"] >= 0.5).astype(int)
daily.to_csv(MK4_DIR / "original_replication_daily_predictions.csv", index=False)
print(f"\nTotal pooled out-of-sample rows: {len(daily)}")

Fold 1: train=183 (purged 1)  test=178  test_seq=178  (2019-12-24 to 2020-08-27)


Fold 2: train=361 (purged 1)  test=178  test_seq=178  (2020-08-28 to 2021-05-04)


Fold 3: train=539 (purged 1)  test=178  test_seq=178  (2021-05-05 to 2022-01-07)


Fold 4: train=717 (purged 1)  test=178  test_seq=178  (2022-01-10 to 2022-09-14)


Fold 5: train=895 (purged 1)  test=178  test_seq=178  (2022-09-15 to 2023-05-22)


Fold 6: train=1073 (purged 1)  test=178  test_seq=178  (2023-05-23 to 2024-01-25)


Fold 7: train=1251 (purged 1)  test=178  test_seq=178  (2024-01-26 to 2024-10-01)


Fold 8: train=1429 (purged 1)  test=178  test_seq=178  (2024-10-02 to 2025-06-06)

Total pooled out-of-sample rows: 1424


## 6. Significance Testing

McNemar's test + fold-clustered bootstrap for each model against the Always-up baseline,
exactly the methodology validated on `btc-direcitonal.ipynb`'s logistic model. Clustering by
fold (not by day) because within a fold every prediction comes from one fitted model -- the
same "too-few-independent-units" caveat applies here too (8 clusters).

In [7]:
def mcnemar(pred_col, actual_col, base_col, df):
    model_correct = (df[pred_col] == df[actual_col]).to_numpy()
    base_correct = (df[base_col] == df[actual_col]).to_numpy()
    b = int(np.sum(model_correct & ~base_correct))
    c = int(np.sum(~model_correct & base_correct))
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else float("nan")
    return b, c, p


def fold_cluster_bootstrap(pred_col, actual_col, base_col, df, n_boot=10000, seed=42):
    d = df.copy()
    d["model_correct"] = (d[pred_col] == d[actual_col]).astype(float)
    d["base_correct"] = (d[base_col] == d[actual_col]).astype(float)
    fold_stats = d.groupby("fold").agg(n=("model_correct", "size"),
                                        model_acc=("model_correct", "mean"),
                                        base_acc=("base_correct", "mean")).reset_index()
    n_folds = len(fold_stats)
    fold_ns = fold_stats["n"].to_numpy()
    fold_model_sum = (fold_stats["model_acc"] * fold_stats["n"]).to_numpy()
    fold_base_sum = (fold_stats["base_acc"] * fold_stats["n"]).to_numpy()
    rng = np.random.default_rng(seed)
    boot_edges = np.empty(n_boot)
    for i in range(n_boot):
        picks = rng.integers(0, n_folds, size=n_folds)
        total_n = fold_ns[picks].sum()
        boot_edges[i] = fold_model_sum[picks].sum() / total_n - fold_base_sum[picks].sum() / total_n
    lo, hi = np.percentile(boot_edges, [2.5, 97.5])
    observed = fold_model_sum.sum() / fold_ns.sum() - fold_base_sum.sum() / fold_ns.sum()
    return n_folds, observed, lo, hi


MODEL_NAMES = {"logit": "Logistic Regression", "rf": "Random Forest",
               "lstm": "LSTM", "cnn_lstm": "CNN-LSTM"}
BASELINE_NAMES = {"always_up_pred": "Always-up"}

summary_rows = []
for model_key, model_label in MODEL_NAMES.items():
    pred_col = f"pred_{model_key}"
    model_acc = (daily[pred_col] == daily["actual"]).mean()
    for base_col, base_label in BASELINE_NAMES.items():
        base_acc = (daily[base_col] == daily["actual"]).mean()
        b, c, p = mcnemar(pred_col, "actual", base_col, daily)
        n_folds, observed, lo, hi = fold_cluster_bootstrap(pred_col, "actual", base_col, daily)
        summary_rows.append({
            "model": model_label, "baseline": base_label,
            "model_acc": model_acc, "baseline_acc": base_acc,
            "edge": observed, "mcnemar_p": p,
            "bootstrap_ci_lo": lo, "bootstrap_ci_hi": hi,
            "ci_excludes_zero": not (lo <= 0 <= hi),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(MK4_DIR / "original_replication_significance_summary.csv", index=False)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
summary

,model,baseline,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,Logistic Regression,Always-up,0.5232,0.5154,0.0077,0.7534,-0.0414,0.0527,False
1,Random Forest,Always-up,0.5316,0.5154,0.0162,0.4426,-0.0330,0.0604,False
2,LSTM,Always-up,0.5000,0.5154,-0.0154,0.4734,-0.0646,0.0281,False
3,CNN-LSTM,Always-up,0.5084,0.5154,-0.0070,0.7425,-0.0218,0.0091,False


## 7. Per-Model, Per-Baseline Readout

In [8]:
for _, row in summary.iterrows():
    sig = "SIGNIFICANT" if row["ci_excludes_zero"] else "not significant"
    model = row["model"]
    baseline = row["baseline"]
    model_acc = row["model_acc"]
    baseline_acc = row["baseline_acc"]
    edge = row["edge"]
    p_val = row["mcnemar_p"]
    ci_lo = row["bootstrap_ci_lo"]
    ci_hi = row["bootstrap_ci_hi"]
    print(f"{model:22s} vs {baseline:12s}  acc={model_acc:.4f}  base={baseline_acc:.4f}  "
          f"edge={edge:+.4f}  p={p_val:.4f}  CI=[{ci_lo:+.4f}, {ci_hi:+.4f}]  ({sig})")


Logistic Regression    vs Always-up     acc=0.5232  base=0.5154  edge=+0.0077  p=0.7534  CI=[-0.0414, +0.0527]  (not significant)
Random Forest          vs Always-up     acc=0.5316  base=0.5154  edge=+0.0162  p=0.4426  CI=[-0.0330, +0.0604]  (not significant)
LSTM                   vs Always-up     acc=0.5000  base=0.5154  edge=-0.0154  p=0.4734  CI=[-0.0646, +0.0281]  (not significant)
CNN-LSTM               vs Always-up     acc=0.5084  base=0.5154  edge=-0.0070  p=0.7425  CI=[-0.0218, +0.0091]  (not significant)


## 8. Summary

**What this replicates:** all 14 of the original notebook's final hand-picked features,
built from that same repo's own CSVs (`bitcoin-price.csv`, `market-data.csv`,
`on-chain-data.csv`, `economic-more.csv`) rather than substitute sources -- including Mean
Coin Age, Miner Reserve, and Open Interest, which an earlier version of this notebook had to
drop for lack of a free data source. It also replicates the original's model comparison
(Logistic Regression, Random Forest, LSTM, CNN-LSTM), rebuilt so every feature is genuinely
leakage-safe and every evaluation is a proper chronological walk-forward with significance
testing -- neither of which the original notebook did (it used a single `TimeSeriesSplit`
last-fold holdout with no significance test).

**Timeframe:** every feature is drawn from the same inner-joined calendar across all four
source files (see §1) -- no feature is forward-filled or extrapolated across a range another
feature doesn't cover, so the model frame's date range in §3 is the true common overlap of
market data (weekdays only), on-chain data (daily), and macro data.

**Purge:** §5 drops the one training row per fold whose label reaches into the test period
(see §5's note). At this 1-day horizon it barely moves the numbers.

**Baseline:** Always-up only (predict "bullish" every day). Persistence (assume today's move
repeats) has been removed as a comparison baseline across this project's notebooks.

**Read the significance table above, not just the raw accuracy column** -- consistent with
every other result this session, a model "winning" on accuracy alone means nothing here; only
`ci_excludes_zero` rows represent a defensible edge, and even those carry the standing caveat
of only 8 bootstrap clusters.